In [ ]:
import itertools
from pathlib import Path
from typing import cast

import numpy as np
import pandas as pd
from IPython.display import display
from numpy.typing import ArrayLike

from utils.corr_assoc_utils import create_corr_assoc_matrix
from utils.misc_utils import crosstab, display_side_by_side, turn_cats_to_continuous, value_counts
from utils.saving_utils import DatasetBundle

pd.options.mode.copy_on_write = True
pd.options.future.infer_string = True


# def display_in_columns(df: pd.DataFrame, n_columns: int = 3):
#     display_side_by_side(
#         *(df.iloc[i * len(df) // n_columns : (i + 1) * len(df) // n_columns] for i in range(n_columns))
#     )

# Hardcoded paths, change to apply to new data
data_full = DatasetBundle.load(Path("data/preprocessed/data_full"))
data_full.meta = cast(pd.DataFrame, data_full.meta)
data_full.na = cast(pd.DataFrame, data_full.na)

data_discov_full = DatasetBundle.load(Path("data/preprocessed/data_discov_full"))
data_discov_full.meta = cast(pd.DataFrame, data_discov_full.meta)
data_discov_full.na = cast(pd.DataFrame, data_discov_full.na)

data_valid_full = DatasetBundle.load(Path("data/preprocessed/data_valid_full"))
data_valid_full.meta = cast(pd.DataFrame, data_valid_full.meta)
data_valid_full.na = cast(pd.DataFrame, data_valid_full.na)

data_discov_reduced = DatasetBundle.load(Path("data/preprocessed/data_discov_reduced"))
data_discov_reduced.meta = cast(pd.DataFrame, data_discov_reduced.meta)
data_discov_reduced.na = cast(pd.DataFrame, data_discov_reduced.na)

data_valid_reduced = DatasetBundle.load(Path("data/preprocessed/data_valid_reduced"))
data_valid_reduced.meta = cast(pd.DataFrame, data_valid_reduced.meta)
data_valid_reduced.na = cast(pd.DataFrame, data_valid_reduced.na)

### variable groups

In [ ]:
na_vals = ["observed", "allowed skip", "not included", "no answer", "imputed skip"]
data_full_na = data_full.na
data_full.meta = data_full.meta.assign(
    **{na_val: lambda x, na_val=na_val: data_full_na[x["name"]].eq(na_val).sum().values for na_val in na_vals}
)
display_side_by_side(
    *(
        data_full.meta.loc[lambda x: x["group"].eq(group), lambda x: x.columns.drop(["group", "labels"])]
        for group in data_full.meta["group"].cat.categories
    ),
    captions=data_full.meta["group"].cat.categories,
)

### look at one variable

In [ ]:
var = "hazar"


def _counts_and_weighted_counts(series: ArrayLike, sample_weight: ArrayLike) -> pd.DataFrame:
    return pd.DataFrame(
        {
            "count": value_counts(series, sample_weight=None, dropna=False),
            "count_weighted": value_counts(series, sample_weight=sample_weight, dropna=False),
        }
    ).sort_index()


print("discovery and validation")
display(data_full.meta[lambda x: x["name"].eq(var)])
display_side_by_side(
    _counts_and_weighted_counts(data_full.data[var], data_full.data["wichtn"]),
    _counts_and_weighted_counts(data_full.na[var], data_full.data["wichtn"]),
)

print("\n")
print("discovery")
if var in data_discov_reduced.data.columns:
    display(data_discov_reduced.meta[lambda x: x["name"].eq(var)])
    display_side_by_side(
        _counts_and_weighted_counts(data_discov_reduced.data[var], data_discov_reduced.data["wichtn"]),
        _counts_and_weighted_counts(data_discov_reduced.na[var], data_discov_reduced.data["wichtn"]),
    )
else:
    print("NOT INCLUDED")

print("\n")
print("validation")
if var in data_valid_reduced.data.columns:
    display(data_valid_reduced.meta[lambda x: x["name"].eq(var)])
    display_side_by_side(
        _counts_and_weighted_counts(data_valid_reduced.data[var], data_valid_reduced.data["wichtn"]),
        _counts_and_weighted_counts(data_valid_reduced.na[var], data_valid_reduced.data["wichtn"]),
    )
else:
    print("NOT INCLUDED")

### search for variable information using regex

In [ ]:
data_full.meta[lambda x: x["name"].str.contains("hnb")]

### show how often two variables are equal and different

In [ ]:
# Check if two variables are equal (only works if both are continuous or categorical with the same labels)
vari, varj = "hnb30", "typ_hnb"

print("discovery and validation")
# True means vari is equal to varj, False means they are different
vars_equal_discov_and_valid = (data_full.data[vari] == data_full.data[varj]) | (
    data_full.data[vari].isna() & data_full.data[varj].isna()
) & (data_full.na[vari] == data_full.na[varj])
display(vars_equal_discov_and_valid.value_counts(normalize=True))

print("\n")
print("discovery")
if vari in data_discov_reduced.data.columns and varj in data_discov_reduced.data.columns:
    # True means vari is equal to varj, False means they are different
    vars_equal_discov = (data_discov_reduced.data[vari] == data_discov_reduced.data[varj]) | (
        data_discov_reduced.data[vari].isna() & data_discov_reduced.data[varj].isna()
    ) & (data_discov_reduced.na[vari] == data_discov_reduced.na[varj])
    display(vars_equal_discov.value_counts(normalize=True))
else:
    print("NOT INCLUDED")

print("\n")
print("validation")
if vari in data_valid_reduced.data.columns and varj in data_valid_reduced.data.columns:
    # True means vari is equal to varj, False means they are different
    vars_equal_valid = (data_valid_reduced.data[vari] == data_valid_reduced.data[varj]) | (
        data_valid_reduced.data[vari].isna() & data_valid_reduced.data[varj].isna()
    ) & (data_valid_reduced.na[vari] == data_valid_reduced.na[varj])
    display(vars_equal_valid.value_counts(normalize=True))
else:
    print("NOT INCLUDED")

In [ ]:
# Show instances where they are not equal
print("discovery and validation")
display(data_full.data.loc[~vars_equal_discov_and_valid, [vari, varj]])

print("\n")
print("discovery")
if vari in data_discov_reduced.data.columns and varj in data_discov_reduced.data.columns:
    display(data_discov_full.data.loc[~vars_equal_discov, [vari, varj]])
else:
    print("NOT INCLUDED")

print("\n")
print("validation")
if vari in data_valid_reduced.data.columns and varj in data_valid_reduced.data.columns:
    display(data_valid_full.data.loc[~vars_equal_valid, [vari, varj]])
else:
    print("NOT INCLUDED")

In [ ]:
# Check if two categorical variables are the same
vari, varj = "hasch_12", "fr12_ha1"
# Use normalize="index" to show percentages for each index


def _crosstab_and_weighted_crosstab(
    series1: ArrayLike, series2: ArrayLike, sample_weight: ArrayLike
) -> tuple[pd.DataFrame, pd.DataFrame]:
    return crosstab(series1, series2, sample_weight=None), crosstab(series1, series2, sample_weight=sample_weight)


print("discovery and validation")
display_side_by_side(
    *_crosstab_and_weighted_crosstab(data_full.data[vari], data_full.data[varj], data_full.data["wichtn"]),
    captions=["counts", "weighted counts"],
)

print("\n")
print("discovery")
if vari in data_discov_reduced.data.columns and varj in data_discov_reduced.data.columns:
    display_side_by_side(
        *_crosstab_and_weighted_crosstab(
            data_discov_reduced.data[vari], data_discov_reduced.data[varj], data_discov_reduced.data["wichtn"]
        ),
        captions=["counts", "weighted counts"],
    )
else:
    print("NOT INCLUDED")

print("\n")
print("validation")
if vari in data_valid_reduced.data.columns and varj in data_valid_reduced.data.columns:
    display_side_by_side(
        *_crosstab_and_weighted_crosstab(
            data_valid_reduced.data[vari], data_valid_reduced.data[varj], data_valid_reduced.data["wichtn"]
        ),
        captions=["counts", "weighted counts"],
    )
else:
    print("NOT INCLUDED")

### Missingness

In [ ]:
# Allowed skips
print("variables where allowed skip was imputed with the negative value:")
display(data_full.na.loc[:, lambda x: x.eq("imputed skip").any()].columns)

print("\n")
print("variables where allowed skip couldn't be imputed:")
display(data_full.na.loc[:, lambda x: x.eq("allowed skip").any()].columns)

In [ ]:
# Variables entirely missing from one dataset
print("discovery all missing:")
display(data_full.na[data_full.data["zeit"].eq("2018")].loc[0:-1, lambda x: ~x.eq("observed").any()].columns)

print("\n")
print("validation all missing:")
display(data_full.na[~data_full.data["zeit"].eq("2018")].loc[0:-1, lambda x: ~x.eq("observed").any()].columns)

In [ ]:
# Variables with highest missingness
observed = ["observed"]
imputed_skip = ["observed", "imputed skip"]
rule_imputed = ["observed", "imputed skip", "rule imputed"]
mice_imputed = ["observed", "imputed skip", "rule imputed", "mice imputed"]

names = ["percent_observed", "percent_observed_weighted"]
cutoff = 0.5
n_min = 5

display_side_by_side(
    *(
        data_discov_reduced.na.isin(x)
        .agg(["mean", lambda x: np.average(x, weights=data_discov_full.data["wichtn"])])
        .transpose()
        .pipe(lambda x: x.rename(columns=dict(zip(x.columns, names, strict=True))))
        .sort_values("percent_observed")
        .loc[lambda x: x.index[(x <= cutoff).any(axis=1)].union(x.index[:n_min], sort=False)]
        for x in [observed, imputed_skip, rule_imputed, mice_imputed]
    ),
    captions=["discovery", "discovery with imputed skips", "discovery rule imputed", "discovery mice imputed"],
)
print("\n")
display_side_by_side(
    *(
        data_valid_reduced.na.isin(x)
        .agg(["mean", lambda x: np.average(x, weights=data_valid_full.data["wichtn"])])
        .transpose()
        .pipe(lambda x: x.rename(columns=dict(zip(x.columns, names, strict=True))))
        .sort_values("percent_observed")
        .loc[lambda x: x.index[(x <= cutoff).any(axis=1)].union(x.index[:n_min], sort=False)]
        for x in [observed, imputed_skip, rule_imputed, mice_imputed]
    ),
    captions=["validation", "validation with imputed skips", "validation rule imputed", "validation mice imputed"],
)

### create table of highest correlations

In [ ]:
cutoff = 0.9


def _make_table(data: DatasetBundle) -> pd.DataFrame:
    if data.meta is None:
        raise ValueError("'data.meta' may not be None.")
    treat_cat_as_continuous: list[str] = data.meta.loc[lambda x: x["type"].isin(["binary", "ordinal"]), "name"].tolist()
    corr_assoc_matrix, p_value_matrix = create_corr_assoc_matrix(
        turn_cats_to_continuous(data.data, treat_cat_as_continuous)
    )
    return (
        pd.DataFrame(
            [
                [vari, varj, corr_assoc, p_value_matrix.loc[vari, varj]]
                for vari, varj in itertools.combinations(data.data.columns, r=2)
                if (corr_assoc := corr_assoc_matrix.loc[vari, varj]) > cutoff
            ]
        )
        .set_axis(["vari", "varj", "corr_assoc", "p_value"], axis=1)
        .sort_values("corr_assoc", ascending=False)
    )


# Show correlation matrices
print("discovery and validation")
display(_make_table(data_full.drop(["lfd", "fall"])))

print("\n")
print("discovery")
display(_make_table(data_discov_reduced))

print("\n")
print("validation")
display(_make_table(data_valid_reduced))

### alcohol variables

In [ ]:
# Look at variable group
data_full.meta[lambda x: x["group"].eq("alcohol")]

In [ ]:
# Ordinal alcohol variable
with pd.option_context("display.max_colwidth", 1000):
    display(data_full.meta[lambda x: x["name"].eq("alk30kat")])

### nicotine variables

In [ ]:
# Look at variable group
data_full.meta[lambda x: x["group"].eq("nicotine")]

In [ ]:
# Check if binary variables are the same
crosstab(data_full.data["hnb30"], data_full.data["typ_hnb"], sample_weight=data_full.data["wichtn"])

### medication variables

In [ ]:
# Look at variable group
data_full.meta[lambda x: x["group"].eq("medication") & ~x["name"].str.contains("_30|^m_|^t_", regex=True)]

### drugs variables

In [ ]:
# Look at variable group
data_full.meta[lambda x: x["group"].eq("drugs")]

In [ ]:
# Check if categorical variables are the same
vars_12 = [
    "hasch_12", "ecsta_12", "lsd_12", "heroi_12", "andop_12", "kokai_12", "crack_12", "pilz_12", "nps_12", "amp_12",
    "meth_12",
]  # fmt: skip
vars_fr12 = [
    "fr12_ha1", "fr12_ec1", "fr12_ls1", "fr12_he1", "fr12_ao1", "fr12_ko1", "fr12_cr1", "fr12_pi1", "fr12_nps1",
    "fr12_au1", "fr12_meth1",
]  # fmt: skip
display_side_by_side(
    *(
        crosstab(data_full.data[vari], data_full.data[varj], sample_weight=data_full.data["wichtn"], dropna=False)
        for vari, varj in zip(vars_12, vars_fr12, strict=True)
    )
)

#### general, socioeconomic, and miscellaneous variables

In [ ]:
data_full.meta[lambda x: ~x["group"].isin(["alcohol", "nicotine", "drugs", "medication"])]